# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [ ]:
#Initial imports

import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json
import pprint as pp
from IPython.display import display

from DbConnector import DbConnector

In [ ]:
# Import datasets
DS_DIR = "movies_ds/"

credits_df = pd.read_csv(DS_DIR + "credits.csv")
keywords_df = pd.read_csv(DS_DIR + "keywords.csv")
links_small_df = pd.read_csv(DS_DIR + "links_small.csv")
links_df = pd.read_csv(DS_DIR + "links.csv")
movies_metadata_df = pd.read_csv(DS_DIR + "movies_metadata.csv")
ratings_small_df = pd.read_csv(DS_DIR + "ratings_small.csv")
ratings_df = pd.read_csv(DS_DIR + "ratings.csv")

In [ ]:
print("Credits: " + str(credits_df.shape))
display(credits_df.head())

print("Keywords: " + str(keywords_df.shape))
display(keywords_df.head())

print("Links_small: " + str(links_small_df.shape))
display(links_small_df.head())

print("Links: " + str(links_df.shape))
display(links_df.head())

print("Movies_metadata: " + str(movies_metadata_df.shape))
display(movies_metadata_df.head())

print("Ratings_small: " + str(ratings_small_df.shape))
display(ratings_small_df.head())

print("Ratings: " + str(ratings_df.shape))
display(ratings_df.head())

- About linear reg for budget based on ratings
- Redundancy for the database structure

# Movies
- **id** (tmdbId)
- adult
- collections
    - name (remove id)
- keywords
    - name (remove id)
- genres
    - name (remove id)
- budget
- original_language
- overview (task 7)
- production_companies
    - name (remove id)
- production_countries
    - iso_3166_1 (remove name)
- release_date
- revenue
- runtime
- spoken_languages (dont need, can maybe remove)
    - iso_639_1 (remove name)
- status
- tagline
- title
- video
- vote_average
- vote_count
- cast
    - contributor_id
    - order
- crew
    - contributor_id
    - job

# Contributors
- **contributor_id**
- name (all people with more names have english and native so could just do english)
- gender (all people with more genders have 0 and either 1 or 2, so prob fine to just ignore 0 here)
- roles
    - tmdbId
    - character
- jobs
    - tmdbId
    - department
    - job

# Raters
- **userId**
- ratings
    - tmdbId (not movieId)
    - rating
    - timestamp

Because average on metadata and the nature of the exercises. Also, since this is static data analysis, we don't have to account for the growing documents.

In [ ]:
movies_metadata_df["budget"] = pd.to_numeric(
    movies_metadata_df["budget"], errors="coerce"
)
movies_metadata_df["budget"] = movies_metadata_df["budget"].astype("Int64")

movies_metadata_df.info()

In [ ]:
import pandas as pd
import ast

def parse_credits(value):
    if pd.isna(value) or value == "":
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

people = []

for _, row in credits_df.iterrows():
    for column in ["cast", "crew"]:
        for person in parse_credits(row[column]):
            people.append({
                "id": person.get("id"),
                "name": person.get("name"),
                "gender": person.get("gender")
            })

people_df = pd.DataFrame(people)

# Remove missing IDs and genders
people_df = people_df.dropna(subset=["id", "gender"])

# Find each person's distinct gender values
gender_details = (
    people_df.groupby(["id", "name"])["gender"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="genders")
)

# Keep only people with multiple gender values
gender_details = gender_details[
    gender_details["genders"].apply(len) > 1
]

print(gender_details.to_string(index=False))

In [ ]:
import pandas as pd
import ast

def parse_credits(value):
    if pd.isna(value) or value == "":
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

people = []

for _, row in credits_df.iterrows():
    for column in ["cast", "crew"]:
        for person in parse_credits(row[column]):
            people.append({
                "id": person.get("id"),
                "name": person.get("name"),
                "role_type": column
            })

people_df = pd.DataFrame(people)

# Remove missing IDs and names
people_df = people_df.dropna(subset=["id", "name"])

# Find people with more than one distinct name
name_counts = (
    people_df.groupby("id")["name"]
    .nunique()
    .reset_index(name="number_of_names")
)

multiple_names = name_counts[name_counts["number_of_names"] > 1]

# Show all names associated with each ID
name_details = (
    people_df[people_df["id"].isin(multiple_names["id"])]
    .groupby("id")["name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index(name="names")
)

name_details["number_of_names"] = name_details["names"].apply(len)

print(name_details.to_string(index=False))

In [ ]:

print("Total movies:", len(movies_metadata_df))

print("Budget = 0:", (movies_metadata_df["budget"] == 0).sum())

for movie in movies_metadata_df["budget"]:
    if movie % 1 != 0:
        print(movie)

print("Budget missing:", movies_metadata_df["budget"].isna().sum())

In [ ]:


df_budget_cleaning = movies_metadata_df[
    (movies_metadata_df["rating_count"].notna())
].copy()

ratingCounts = ratings_df["movieId"].value_counts(dropna=False).reset_index()
ratingCounts.columns = ["movieId", "count"]



df_budget_cleaning = df_budget_cleaning[]

df_clean["MAX_SPEED_KMH"] = (
    df_clean["POLYLINE"]
    .apply(json.loads)
    .apply(max_speed_kmh)
)

df = movies_metadata_df[
    (movies_metadata_df["rating_count"].notna())
].copy()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(
    df["rating_count"],
    df["budget"],
    alpha=0.3
)

plt.xlabel("Number of ratings")
plt.ylabel("Budget")
plt.title("Movie budget vs number of ratings")

plt.show()

In [ ]:
plt.scatter(
    df["rating_count"],
    df["budget"],
    alpha=0.3
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Number of ratings (log scale)")
plt.ylabel("Budget (log scale)")

plt.title("Movie budget vs rating count")

plt.show()

In [ ]:
print(df[["budget", "rating_count"]].corr())


df["log_budget"] = np.log1p(df["budget"])

df["log_rating_count"] = np.log1p(
    df["rating_count"]
)

print(
    df[["log_budget", "log_rating_count"]]
    .corr()
)

In [ ]:
from pymongo import MongoClient, version

class DbConnector:
    """
    Connects to the MongoDB server on the Ubuntu virtual machine.
    Connector needs HOST, USER and PASSWORD to connect.

    Example:
    HOST = "tdt4225-00.idi.ntnu.no" // Your server IP address/domain name
    USER = "testuser" // This is the user you created and added privileges for
    PASSWORD = "test123" // The password you set for said user
    """

    def __init__(self,
                 DATABASE='my_db',
                 PORT=27018,
                 HOST="localhost",
                 USER="groupuser",
                 PASSWORD="WaterisGood02"):
        uri = "mongodb://%s:%s@%s:%d/%s?authSource=%s" % (USER, PASSWORD, HOST, PORT, DATABASE, DATABASE)
        # Connect to the databases
        try:
            self.client = MongoClient(uri)
            self.db = self.client[DATABASE]
        except Exception as e:
            print("ERROR: Failed to connect to db:", e)

        # get database information
        print("You are connected to the database:", self.db.name)
        print("-----------------------------------------------\n")

    def close_connection(self):
        # close the cursor
        # close the DB connection
        self.client.close()
        print("\n-----------------------------------------------")
        print("Connection to %s-db is closed" % self.db.name)

In [ ]:
from pprint import pprint 
from DbConnector import DbConnector

class Program:

    def __init__(self):
        self.connection = DbConnector()
        self.client = self.connection.client
        self.db = self.connection.db

    def create_coll(self, collection_name):
        collection = self.db.create_collection(collection_name)    
        print('Created collection: ', collection)

    def insert_documents(self, docs, collection_name):
        collection = self.db[collection_name]
        collection.insert_many(docs)
        
    def fetch_documents(self, collection_name):
        collection = self.db[collection_name]
        documents = collection.find({})
        for doc in documents: 
            pprint(doc)
        

    def drop_coll(self, collection_name):
        collection = self.db[collection_name]
        collection.drop()

        
    def show_coll(self):
        collections = self.client['test'].list_collection_names()
        print(collections)
         


def main():
    program = None
    try:
        program = Program()
        program.drop_coll(collection_name="Movies")
        program.drop_coll(collection_name="People")
        program.drop_coll(collection_name="Ratings")

        program.create_coll(collection_name="Movies")
        program.create_coll(collection_name="People")
        program.create_coll(collection_name="Ratings")
        program.show_coll()
        
        program.insert_documents(collection_name="Person")
        program.fetch_documents(collection_name="Person")
        program.drop_coll(collection_name="Person")

        # Check that the table is dropped
        program.show_coll()
    except Exception as e:
        print("ERROR: Failed to use database:", e)
    finally:
        if program:
            program.connection.close_connection()


if __name__ == '__main__':
    #main()
    print("Done")